In [58]:
pip install --upgrade torch ultralytics

  Using cached torch-2.7.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.5.1.17-py3-none-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.3.0.4-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.7.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.7.1.2-py3-none-manylinux2014_x8

class C3k2(C2f):

    def __init__(
        self, c1: int, c2: int, n: int = 1, c3k: bool = False, e: float = 0.5, g: int = 1, shortcut: bool = True
    ):
        super().__init__(c1, c2, n, shortcut, g, e)
        self.m = nn.ModuleList(
            C3k(self.c, self.c, 2, shortcut, g) if c3k else Bottleneck(self.c, self.c, shortcut, g) for _ in range(n)
        )


class C3k(C3):

    def __init__(self, c1: int, c2: int, n: int = 1, shortcut: bool = True, g: int = 1, e: float = 0.5, k: int = 3):

        super().__init__(c1, c2, n, shortcut, g, e)
        c_ = int(c2 * e) 
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, k=(k, k), e=1.0) for _ in range(n)))

                              nn.BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              nn.Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1), bias=False),
                              nn.BatchNorm2d(384, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              
                              nn.Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              nn.Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              nn.Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), bias=False),
                              nn.BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              
                              nn.Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              nn.Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                             
                              nn.Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True),
                              nn.Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False),
                              nn.BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True),
                              nn.SiLU(inplace=True))

In [59]:
from ultralytics import YOLO
import os, pandas, numpy, cv2
from pathlib import Path
from PIL import Image
import torch.nn as nn

In [60]:
CLASSES=['labels']

yaml_content = f"""
train: /kaggle/input/simulated-object-data/data/train/images
val: /kaggle/input/simulated-object-data/data/val/images

nc: {len(CLASSES)}
names: {CLASSES}
"""

with open("dataset.yaml", "w") as f:
    f.write(yaml_content)

print("dataset.yaml created!")

dataset.yaml created!


In [61]:
class Conv(nn.Module):
    default_act = nn.LeakyReLU(inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def forward_fuse(self, x):
        return self.act(self.conv(x))
        
class RE_Conv(nn.Module):
    default_act = nn.LeakyReLU(inplace=True)

    def __init__(self, c1, c2, kernel_size=1, stride=1, padding=None, act=True, eps = 0.001, momentum = 0.03):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False)
        #self.bn = nn.BatchNorm2d(c2, eps, momentum)
        self.act = self.default_act if act is True else act if isinstance(act, nn.Module) else nn.Identity()

    def forward(self, x):
        return self.act(self.conv(x))

    def forward_fuse(self, x):
        return self.act(self.conv(x))

In [62]:
yolo = YOLO("yolo11x.pt")
yolo

100%|██████████| 109M/109M [00:00<00:00, 291MB/s] 


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(96, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(96, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(192, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(384, eps=0.001, momentum=0.03, affine=True, track_

In [63]:
yolo.model.model[2]

C3k2(
  (cv1): Conv(
    (conv): Conv2d(192, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(384, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0-1): 2 x C3k(
      (cv1): Conv(
        (conv): Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(48, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv3): Conv(
        (conv): Conv2d(96, 96, ker

In [65]:
class C3k2(nn.Module):
    def __init__(self, x_0):
        super().__init__()

        self.cv1 = RE_Conv(x_0, x_0, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))
        self.cv2 = RE_Conv(x_0*2, x_0*2, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1))

        self.m = nn.ModuleList([
                      RE_Conv(x_0//2, x_0//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      RE_Conv(x_0//2, x_0//4, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                      RE_Conv(x_0//2, x_0//2, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1)),
                
                      nn.Sequential(
                          RE_Conv(x_0//2, x_0//2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          RE_Conv(x_0//2, x_0//2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                      
                          RE_Conv(x_0//2, x_0//2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
                          RE_Conv(x_0//2, x_0//2, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
                      )]
                  )
    def forward(self, x):
        x = self.cv1(x)
        x = self.cv2(x)
        return self.m(x)

In [66]:
C3k2(192)

C3k2(
  (cv1): RE_Conv(
    (conv): Conv2d(192, 192, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1), bias=False)
    (act): LeakyReLU(negative_slope=0.01, inplace=True)
  )
  (cv2): RE_Conv(
    (conv): Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1), bias=False)
    (act): LeakyReLU(negative_slope=0.01, inplace=True)
  )
  (m): ModuleList(
    (0-1): 2 x RE_Conv(
      (conv): Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1), bias=False)
      (act): LeakyReLU(negative_slope=0.01, inplace=True)
    )
    (2): RE_Conv(
      (conv): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1), padding=(1, 1), bias=False)
      (act): LeakyReLU(negative_slope=0.01, inplace=True)
    )
    (3): Sequential(
      (0): RE_Conv(
        (conv): Conv2d(96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (act): LeakyReLU(negative_slope=0.01, inplace=True)
      )
      (1): RE_Conv(
        (conv): Conv2d(96, 96, kernel_size=(3, 3), stride=

In [67]:
yolo.model.model[6]

C3k2(
  (cv1): Conv(
    (conv): Conv2d(768, 768, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(768, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(1536, 768, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(768, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0-1): 2 x C3k(
      (cv1): Conv(
        (conv): Conv2d(384, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(384, 192, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(192, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv3): Conv(
        (conv): Conv2d(384,

In [68]:
yolo.model.model[0] = Conv(3, 96, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[1] = Conv(96, 192, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[2] = C3k2(192)
yolo.model.model[3] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[5] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[7] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[17] = Conv(384, 384, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
yolo.model.model[20] = Conv(768, 768, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))

In [ ]:
yolo.train( data='/kaggle/working/dataset.yaml',
            epochs=150,
            batch=10,
            imgsz=724,
            patience=5,
            lr0=0.001,
            lrf=0.02,
            optimizer="SGD",
            momentum=0.96,
            weight_decay=0.001,
            cos_lr=True,
            dropout=0.3,
            label_smoothing=0.1,
            mosaic=0.5,
            mixup=0.15,
            copy_paste=0.1,
            fliplr=0.5,
            flipud=0.5,
            hsv_h=0.5,
            hsv_s=0.9,
            hsv_v=0.9,
            translate=0.2,
            scale=0.5,
            shear=0.2,
            perspective=0.0002,
            val=True,
            workers=8,
            seed=42,
            device=[-1, -1]
        )
valid_results = yolo.val()
print(valid_results)

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in in the future.
Searching for 2 idle GPUs with free memory >= 20.0% and free utilization >= 0.0%...
Selected idle CUDA devices [1, 0]
Ultralytics 8.3.159 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:1 (Tesla T4, 15095MiB)
                                                        CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=1,0, dfl=1.5, dnn=False, dropout=0.3, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.5, hsv_s=0.9, hsv_v=0.9, imgsz=724, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lr

100%|██████████| 5.35M/5.35M [00:00<00:00, 73.0MB/s]


AMP: checks passed ✅
WARNING ⚠️ imgsz=[724] must be multiple of max stride 32, updating to [736]
train: Fast image access ✅ (ping: 0.7±1.0 ms, read: 1100.1±566.7 MB/s, size: 2780.0 KB)


train: Scanning /kaggle/input/simulated-object-data/data/train/labels... 10 images, 0 backgrounds, 0 corrupt:  18%|█▊        | 10/56 [00:00<00:00, 82.46it/s]

In [ ]:
model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')

In [ ]:
output_dir = r"/kaggle/working/predictions/labels"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
for i in os.listdir('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images'):
    img_path = f'/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images/{i}'
    results = model.predict(img_path, 
                            conf=0.3, device=0, verbose=False) # 0 - GPU or "cpu" Image.fromarray(cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2HSV))
    output_txt = f"{output_dir}/{i.split('.')[0]}.txt"

    with open(output_txt, "w") as f:
        found = False
        for result in results:
            img_height, img_width = result.orig_shape
            boxes = result.boxes.data

            if boxes is None or len(boxes) == 0:
                continue

            filtered_boxes = boxes[boxes[:, 4] >= 0.05]
            if len(filtered_boxes) == 0:
                continue

            found = True
            for box in filtered_boxes:
                x1, y1, x2, y2, confidence, cls_id = box.tolist()

                x_center = ((x1 + x2) / 2) / img_width
                y_center = ((y1 + y2) / 2) / img_height
                width = (x2 - x1) / img_width
                height = (y2 - y1) / img_height

                f.write(f"0 {confidence:.6f} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        if not found:
            f.write("")

In [ ]:
rows = []
output_dir = Path("/kaggle/working/predictions/labels")
TEST = Path('/kaggle/input/multi-instance-object-detection-challenge/Starter_Dataset/TestImages/images')
test_imgs = {p.stem for p in TEST.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
predicted = set()

for file in output_dir.glob("*.txt"):
    name = file.stem
    predicted.add(name)

    try:
        lines = [l.strip() for l in open(file) if len(l.strip().split()) == 6]
    except:
        lines = []

    rows.append({"image_id": name, "prediction_string": " ".join(lines) if lines else "no boxes"})

for name in test_imgs - predicted:
    rows.append({"image_id": name, "prediction_string": "no boxes"})

work_dir = '/kaggle/working'

for filename in os.listdir(work_dir):
    file_path = os.path.join(work_dir, filename)

    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)
    except Exception as e:
        print(f'Error file: {file_path}. Cause: {e}')

rows = pandas.DataFrame(rows)
rows.to_csv("submission.csv", index=False)
rows